<a href="https://colab.research.google.com/github/Theresa-Gao-0529/MSSP607/blob/main/Week_6_Lambda_Function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Participation Activity Week 6: Lambda Function for a DataFrame



## 1. Import pandas and load the data


In [ ]:
from pathlib import Path
from urllib.parse import quote
import pandas as pd

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# GitHub repository that contains the CSV file.
owner = "Theresa-Gao-0529"
repository = "MSSP607"
repo_url = f"https://github.com/{owner}/{repository}"
repo_path = Path(".") / repository
data_relative_path = Path("data") / "week6" / "Student Performance Data.csv"
github_data_url = (
    f"https://raw.githubusercontent.com/{owner}/{repository}/main/"
    f"{quote(data_relative_path.as_posix())}"
)

# Read from GitHub first. This does not depend on whether the notebook is in Colab.
DATA_FILE = github_data_url
try:
    students = pd.read_csv(DATA_FILE)
except Exception as github_error:
    # Optional fallback for offline/local execution.
    local_copy = Path("Student Performance Data.csv")
    if not local_copy.exists():
        raise FileNotFoundError(
            f"Could not read the dataset from GitHub ({DATA_FILE}) or find {local_copy}."
        ) from github_error
    DATA_FILE = local_copy
    students = pd.read_csv(DATA_FILE)

print(f"Loaded data from: {DATA_FILE}")
print(f"Dataset shape: {students.shape[0]} rows × {students.shape[1]} columns")
display(students.head())

## 2. Define the score columns



In [ ]:
score_columns = ["math score", "reading score", "writing score"]
required_columns = ["gender", "test preparation course", *score_columns]
missing_columns = [column for column in required_columns if column not in students.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

students[score_columns].describe()

## 3. Create a new DataFrame with a lambda function


In [ ]:
performance_summary = lambda df: (
    df.loc[:, ["gender", "test preparation course", *score_columns]]
    .copy()
    .assign(
        average_score=lambda summary: summary[score_columns].mean(axis=1).round(2)
    )
    .assign(
        performance_level=lambda summary: summary["average_score"].apply(
            lambda score: (
                "High" if score >= 80
                else "Satisfactory" if score >= 60
                else "Needs support"
            )
        )
    )
)

summary = performance_summary(students)
print(f"New DataFrame shape: {summary.shape[0]} rows × {summary.shape[1]} columns")
display(summary.head(10))

## 4. Review and save the result



In [ ]:
display(summary["performance_level"].value_counts().rename_axis("performance_level").to_frame("student_count"))

OUTPUT_FILE = Path("Student Performance Summary.csv")
summary.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(summary)} rows to {OUTPUT_FILE}")

if IN_COLAB:
    files.download(str(OUTPUT_FILE))